# Notebook 2 - CO2 Estimation

Loads the route panel, fetches aircraft-type data from Eurostat, estimates route-level CO2 using fuel-burn lookups and rerouting multipliers, and saves updated panel for analysis.

## 1. Imports and directories

In [ ]:
import pandas as pd
import numpy as np
import requests
import warnings
from io import StringIO
from pathlib import Path
from linearmodels.panel import PanelOLS
import matplotlib.pyplot as plt
import gzip
import shutil
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 150, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})
C = {'primary':'#1A4A8A','secondary':'#C0392B','neutral':'#888780','highlight':'#2DBDB6'}

RAW_DIR   = Path.home() / 'econ62020' / 'raw'
CLEAN_DIR = Path.home() / 'econ62020' / 'clean'
OUTPUT_DIR = CLEAN_DIR

def haversine_vec(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

## 2. Load route panel

In [2]:
route_panel_nc = pd.read_parquet(CLEAN_DIR / 'route_panel_nc.parquet')
df_pivot       = pd.read_parquet(CLEAN_DIR / 'route_panel_full.parquet')
print(f'route_panel_nc: {route_panel_nc.shape}')
print(f'df_pivot:       {df_pivot.shape}')

route_panel_nc: (783057, 31)
df_pivot:       (885377, 31)


## 3. Fuel-burn and CO2 constants

In [3]:
CO2_PER_KG_FUEL = 3.16

# Distance-band fuel burn (kg per flight) — used for baseline CO2 when aircraft-type data is unavailable
FUEL_PER_FLIGHT_DIST = {
    'short':     3000,
    'medium':    8000,
    'long':     30000,
    'very_long': 50000,
}

def get_distance_band(dist_km):
    if pd.isna(dist_km):    return np.nan
    elif dist_km < 1000:    return 'short'
    elif dist_km < 3000:    return 'medium'
    elif dist_km < 7000:    return 'long'
    else:                   return 'very_long'

# Aircraft-type fuel burn (kg per flight)
FUEL_BURN_KG = {
    'AC_319': 5200,  'AC_320': 5800,  'AC_321': 6500,
    'AC_S20': 5800,  'AC_737': 5500,  'AC_73H': 6200,
    'AC_73G': 5800,  'AC_73J': 6500,  'AC_757': 8500,
    'AC_318': 4800,  'AC_717': 4500,  'AC_A32': 5800,
    'AC_B46': 5800,
    'AC_767': 18000, 'AC_777': 35000, 'AC_77W': 38000,
    'AC_787': 25000, 'AC_788': 23000, 'AC_789': 26000,
    'AC_330': 20000, 'AC_332': 19000, 'AC_333': 21000,
    'AC_340': 28000, 'AC_350': 24000, 'AC_380': 55000,
    'AC_744': 42000, 'AC_748': 45000, 'AC_747': 42000,
    'AC_300': 20000, 'AC_310': 18000, 'AC_M11': 38000,
    'AC_DC1': 15000, 'AC_DC8': 20000, 'AC_L10': 25000,
    'AC_E13':  2800, 'AC_E14':  3000, 'AC_E17':  3500,
    'AC_E19':  3800, 'AC_E75':  3200, 'AC_CR2':  2200,
    'AC_CR7':  2800, 'AC_CR9':  3200, 'AC_F10':  4500,
    'AC_F50':  2500, 'AC_DH8':  2000, 'AC_F70':  3800,
    'AC_E12':  2600, 'AC_RJ1':  3000, 'AC_RJ8':  2800,
    'AC_AT4':  1200, 'AC_AT7':  1500, 'AC_DO3':   900,
    'AC_C60':  1800, 'AC_S34':  2200,
    'AC_727': 12000, 'AC_DC9':  5000, 'AC_M82':  6000,
    'AC_M83':  6200, 'AC_M87':  6000, 'AC_M88':  6200,
    'AC_M90':  6200, 'AC_M80':  5800, 'AC_M81':  5800,
    'AC_T13':  1000, 'AC_T15':  2500,
    'AC_BN2':   400, 'AC_AN3':   300,
    'AC_OEM':  4000, 'AC_OAB':  5800, 'AC_OCA': 25000,
    'AC_OBA':  8000, 'AC_OBO': 18000, 'AC_OAN':  3500,
    'AC_ODO':  1500, 'AC_OAS':  3500, 'AC_ODH':  2000,
    'AC_OUT':  5800, 'AC_OLO':  4000, 'AC_OAL':  5800,
    'AC_ODG':  1500, 'AC_OFK':  3800, 'AC_OSA':  3500,
    'UNK':     5800,
}

## 4. Load and process avia_tf_aca (aircraft-type data)

Monthly commercial flights by aircraft type from Eurostat.

In [4]:
fpath = RAW_DIR / 'estat_avia_tf_aca.tsv'

if not fpath.exists():
    import requests, gzip, shutil
    print("avia_tf_aca not found — downloading from Eurostat...")
    url = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/avia_tf_aca/?format=TSV&compressed=true"
    gz_path = RAW_DIR / 'estat_avia_tf_aca.tsv.gz'
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(gz_path, 'wb') as f:
            shutil.copyfileobj(r.raw, f)
    with gzip.open(gz_path, 'rb') as f_in, open(fpath, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    gz_path.unlink()
    print(f"Saved to {fpath}")

headers  = pd.read_csv(fpath, sep='\t', nrows=0)
all_cols = headers.columns.tolist()
key_col  = all_cols[0]

keep_cols = [key_col] + [
    c for c in all_cols
    if str(c).strip().count('-') == 1
    and len(str(c).strip()) == 7
    and not str(c).strip().endswith(('Q1','Q2','Q3','Q4'))
    and int(str(c).strip()[:4]) >= 2015
]
print(f'Keeping {len(keep_cols)} of {len(all_cols)} columns')

df_aca_raw = pd.read_csv(
    fpath, sep='\t', usecols=keep_cols,
    na_values=[':', ': ', 'z', 'Z'], engine='pyarrow'
)
print(f'Raw shape: {df_aca_raw.shape}')

key_parts              = df_aca_raw[key_col].str.strip().str.split(',', expand=True)
df_aca_raw['FREQ']         = key_parts[0].str.strip()
df_aca_raw['UNIT']         = key_parts[1].str.strip()
df_aca_raw['TRA_MEAS']     = key_parts[2].str.strip()
df_aca_raw['TRA_COV']      = key_parts[3].str.strip()
df_aca_raw['AIRCRAFT']     = key_parts[4].str.strip()
df_aca_raw['REP_AIRP_RAW'] = key_parts[5].str.strip()
df_aca_raw['REP_AIRP']     = df_aca_raw['REP_AIRP_RAW'].str.split('_').str[-1]

orig_airports = set(df_pivot['ORIG_ICAO'].dropna().unique())

df_aca = df_aca_raw[
    (df_aca_raw['FREQ']     == 'M') &
    (df_aca_raw['TRA_MEAS'] == 'CAF') &
    (df_aca_raw['TRA_COV']  == 'INTL') &
    (df_aca_raw['REP_AIRP'].isin(orig_airports))
].copy()

print(f'After filtering: {df_aca.shape}')

avia_tf_aca not found — downloading from Eurostat...
Saved to /Users/trinityrose/econ62020/test/raw/estat_avia_tf_aca.tsv
Keeping 136 of 294 columns
Raw shape: (3124982, 136)
After filtering: (24278, 143)


## 5. Melt and map fuel burns

In [5]:
time_cols = [col for col in df_aca.columns
             if str(col).strip().count('-') == 1
             and len(str(col).strip()) == 7
             and not str(col).strip().endswith(('Q1','Q2','Q3','Q4'))]

df_aca_long = df_aca.melt(
    id_vars=['AIRCRAFT','REP_AIRP'],
    value_vars=time_cols,
    var_name='TIME_STR', value_name='FLIGHTS_AC'
)
df_aca_long['TIME_STR'] = df_aca_long['TIME_STR'].str.strip()
df_aca_long['YEAR']     = df_aca_long['TIME_STR'].str[:4].astype(int)
df_aca_long['MONTH']    = df_aca_long['TIME_STR'].str[5:7].astype(int)
df_aca_long['FLIGHTS_AC'] = pd.to_numeric(
    df_aca_long['FLIGHTS_AC'].astype(str)
    .str.replace(r'[a-zA-Z\s]','',regex=True).replace('',np.nan),
    errors='coerce')
df_aca_long = df_aca_long.dropna(subset=['FLIGHTS_AC'])
df_aca_long = df_aca_long[df_aca_long['FLIGHTS_AC'] > 0]
df_aca_long = df_aca_long[~df_aca_long['AIRCRAFT'].isin(['TOTAL','OTH'])].copy()
print(f'Long format shape: {df_aca_long.shape}')

df_aca_long['FUEL_PER_FLIGHT'] = df_aca_long['AIRCRAFT'].map(FUEL_BURN_KG)
mapped   = df_aca_long['FUEL_PER_FLIGHT'].notna()
unmapped = df_aca_long[~mapped]['AIRCRAFT'].value_counts()
print(f'Mapped: {mapped.sum():,} ({100*mapped.mean():.1f}%)')

df_aca_long['CO2_KG'] = (
    df_aca_long['FLIGHTS_AC'] * df_aca_long['FUEL_PER_FLIGHT'] * CO2_PER_KG_FUEL
)

Long format shape: (355083, 6)
Mapped: 355,083 (100.0%)


## 6. Aggregate to airport-month level

In [8]:
airport_co2 = (
    df_aca_long
    .dropna(subset=['CO2_KG'])
    .groupby(['REP_AIRP','YEAR','MONTH'])
    .agg(
        TOTAL_CO2_KG     = ('CO2_KG',     'sum'),
        TOTAL_FLIGHTS_AC = ('FLIGHTS_AC', 'sum'),
    )
    .reset_index()
)
airport_co2['AVG_FUEL_PER_FLIGHT'] = (
    airport_co2['TOTAL_CO2_KG'] / CO2_PER_KG_FUEL / airport_co2['TOTAL_FLIGHTS_AC']
)
print(f'Airport-month shape: {airport_co2.shape}')
print(f'Airports covered:    {airport_co2["REP_AIRP"].nunique():,}')

# Check using Helsinki
print('\nHelsinki 2019 check:')
print(airport_co2[
    (airport_co2['REP_AIRP'] == 'EFHK') & (airport_co2['YEAR'] == 2019)
].sort_values('MONTH')[['MONTH','TOTAL_FLIGHTS_AC','AVG_FUEL_PER_FLIGHT']].to_string(index=False))

Airport-month shape: (39458, 6)
Airports covered:    367

Helsinki 2019 check:
 MONTH  TOTAL_FLIGHTS_AC  AVG_FUEL_PER_FLIGHT
     1           10273.0          6233.028327
     2            9280.0          6187.478448
     3           10487.0          6226.556689
     4           10943.0          6122.032349
     5           12036.0          6104.893652
     6           12018.0          6113.288401
     7           12211.0          6193.735157
     8           12208.0          6187.860419
     9           11821.0          6128.297098
    10           11744.0          6213.717643
    11           10165.0          6203.649779
    12           10188.0          6274.862583


## 7. Exposure groups and rerouting multipliers

Routes are grouped by European origin and Asian destination. Multipliers reflect average great-circle distance ratios post-closure vs pre-war baseline from Eurostat snapshot 29.

- **high**: Finland → East Asia (worst affected, longest detour)
- **medium**: Other Europe → East Asia
- **low**: Europe → South/Southeast/Central Asia/Middle East
- **none**: unaffected routes

In [11]:
# Europe origin groups
FINLAND             = {'FI'}
SCANDINAVIA         = {'DK','SE','NO','IS'}
CENTRAL_WESTERN_EU  = {'DE','NL','BE','FR','GB','IE','AT','CH','LU','CZ','PL'}
SOUTHERN_EU         = {'ES','PT','IT','GR','MT','CY'}
EASTERN_EU          = {'EE','LV','LT','HU','SK','SI','HR','RO','BG','RS','BA','ME','MK','AL','MD'}
EUROPE_ALL          = FINLAND | SCANDINAVIA | CENTRAL_WESTERN_EU | SOUTHERN_EU | EASTERN_EU

# Asia destination groups
EAST_ASIA      = {'CN','JP','KR','HK','TW','MN'}
SOUTH_ASIA     = {'IN','PK','BD','LK','NP','MV'}
SOUTHEAST_ASIA = {'SG','TH','VN','MY','ID','PH','BN'}
CENTRAL_ASIA   = {'KZ','UZ','KG','TJ','TM'}
MIDDLE_EAST    = {'AE','QA','SA','OM','BH','KW','IL','JO','LB','IQ','IR'}
ASIA_ALL       = EAST_ASIA | SOUTH_ASIA | SOUTHEAST_ASIA | CENTRAL_ASIA | MIDDLE_EAST

def exposure_group(row):
    if row['POST'] == 0 or row['AFFECTED'] == 0:
        return 'none'
    a, b = row['ORIG_CC'], row['DEST_CC']
    if a in EUROPE_ALL and b in ASIA_ALL:
        eu, asia = a, b
    elif b in EUROPE_ALL and a in ASIA_ALL:
        eu, asia = b, a
    else:
        return 'none'
    if eu in FINLAND and asia in EAST_ASIA:
        return 'high'
    if asia in EAST_ASIA:
        return 'medium'
    if asia in (SOUTH_ASIA | SOUTHEAST_ASIA | MIDDLE_EAST | CENTRAL_ASIA):
        return 'low'
    return 'low'

MULTIPLIERS = {'none': 1.00, 'low': 1.01, 'medium': 1.16, 'high': 1.43}

## 8. Estimate route-level CO2

Baseline CO2 uses distance-band fuel averages. Rerouting-adjusted CO2 scales the baseline by the exposure-group multiplier post-closure.

In [12]:
df_co2 = df_pivot.copy()

df_co2['EXPOSURE_GROUP'] = df_co2.apply(exposure_group, axis=1)
df_co2['REROUTE_RATIO']  = df_co2['EXPOSURE_GROUP'].map(MULTIPLIERS)
df_co2['ADJUSTED_DISTANCE'] = df_co2['ROUTE_DIST_KM'] * df_co2['REROUTE_RATIO']

df_co2['DIST_BAND']       = df_co2['ROUTE_DIST_KM'].apply(get_distance_band)
df_co2['FUEL_PER_FLIGHT'] = df_co2['DIST_BAND'].map(FUEL_PER_FLIGHT_DIST)

# Baseline CO2 (original route distance)
df_co2['CO2_KG_BASE_EST'] = df_co2['FLIGHTS'] * df_co2['FUEL_PER_FLIGHT'] * CO2_PER_KG_FUEL

# Rerouting-adjusted CO2
df_co2['CO2_KG_ADJ_EST']  = df_co2['CO2_KG_BASE_EST'] * df_co2['REROUTE_RATIO']

# Emissions exposure: flights × adjusted distance
df_co2['EMISSIONS_EXPOSURE'] = df_co2['FLIGHTS'] * df_co2['ADJUSTED_DISTANCE']

# Log outcomes
df_co2['LOG_FLIGHTS']            = np.log(df_co2['FLIGHTS'].replace(0, np.nan))
df_co2['LOG_PASSENGERS']         = np.log(df_co2['PASSENGERS'].replace(0, np.nan))
df_co2['LOG_SEATS']              = np.log(df_co2['SEATS'].replace(0, np.nan))
df_co2['LOG_CO2_ADJ_EST']        = np.log(df_co2['CO2_KG_ADJ_EST'].replace(0, np.nan))
df_co2['LOG_EMISSIONS_EXPOSURE'] = np.log(df_co2['EMISSIONS_EXPOSURE'].replace(0, np.nan))

# log(1+x) outcomes preserving zeros
df_co2['LOG1P_FLIGHTS']            = np.log1p(df_co2['FLIGHTS'])
df_co2['LOG1P_CO2_ADJ_EST']        = np.log1p(df_co2['CO2_KG_ADJ_EST'])
df_co2['LOG1P_EMISSIONS_EXPOSURE'] = np.log1p(df_co2['EMISSIONS_EXPOSURE'])

df_co2['LOAD_FACTOR'] = (df_co2['PASSENGERS'] / df_co2['SEATS']).replace([np.inf,-np.inf], np.nan).clip(0,1)

print(f'Baseline CO2 estimated for {df_co2["CO2_KG_BASE_EST"].notna().sum():,} rows')
print(f'Adjusted CO2 estimated for {df_co2["CO2_KG_ADJ_EST"].notna().sum():,} rows')

Baseline CO2 estimated for 868,498 rows
Adjusted CO2 estimated for 868,498 rows


### Exposure group distribution

In [13]:
print(df_co2['EXPOSURE_GROUP'].value_counts(dropna=False))
print()
print(
    df_co2
    .groupby('EXPOSURE_GROUP')
    .agg(
        routes      = ('AIRP_PR',      'nunique'),
        obs         = ('AIRP_PR',      'size'),
        flights     = ('FLIGHTS',      'sum'),
        mean_dist   = ('ROUTE_DIST_KM','mean'),
        mean_mult   = ('REROUTE_RATIO','mean'),
    )
    .sort_values('mean_mult')
)

EXPOSURE_GROUP
none      881215
low         2464
medium      1555
high         143
Name: count, dtype: int64

                routes     obs      flights    mean_dist  mean_mult
EXPOSURE_GROUP                                                     
none             14346  881215  107406593.0  1775.298307       1.00
low                 86    2464     185149.0  6698.802112       1.01
medium              74    1555     104776.0  8853.280544       1.16
high                 6     143       7852.0  7548.752386       1.43


## 9. Build route_panel_nc with CO2

Excludes COVID years (2020–2021) and GB post-2019. Rebuilds DiD treatment variables.

In [14]:
route_panel_nc = df_co2[~df_co2['YEAR'].isin([2020, 2021])].copy()
route_panel_nc = route_panel_nc[
    ~((route_panel_nc['ISO2'] == 'GB') & (route_panel_nc['YEAR'] >= 2020))
].copy()

route_panel_nc['POST'] = (
    ((route_panel_nc['YEAR'] == 2022) & (route_panel_nc['MONTH'] >= 2)) |
    (route_panel_nc['YEAR'] >= 2023)
).astype(int)
route_panel_nc['TIME'] = route_panel_nc['YEAR'] * 100 + route_panel_nc['MONTH']

route_panel_nc['DID']         = route_panel_nc['AFFECTED'].astype(int) * route_panel_nc['POST']
route_panel_nc['DID_RUSSIA']  = (route_panel_nc['HITS_RUSSIA']  & ~route_panel_nc['HITS_UKRAINE']).astype(int) * route_panel_nc['POST']
route_panel_nc['DID_UKRAINE'] = (route_panel_nc['HITS_UKRAINE'] & ~route_panel_nc['HITS_RUSSIA']).astype(int)  * route_panel_nc['POST']
route_panel_nc['DID_BOTH']    = (route_panel_nc['HITS_RUSSIA']  &  route_panel_nc['HITS_UKRAINE']).astype(int) * route_panel_nc['POST']

# Continuous treatment intensity: rerouting penalty × POST
route_panel_nc['REROUTE_PENALTY'] = route_panel_nc['REROUTE_RATIO'] - 1
route_panel_nc['DID_INTENSITY']   = route_panel_nc['REROUTE_PENALTY'] * route_panel_nc['POST']

print(f'route_panel_nc: {route_panel_nc.shape}')
print(f'Countries:       {route_panel_nc["ISO2"].nunique()}')
print(f'Routes:          {route_panel_nc["AIRP_PR"].nunique():,}')
print(f'Affected routes: {route_panel_nc[route_panel_nc["AFFECTED"]]["AIRP_PR"].nunique():,}')
print(f'POST obs:        {route_panel_nc["POST"].sum():,}')
print(f'PRE obs:         {(1-route_panel_nc["POST"]).sum():,}')

route_panel_nc: (783057, 46)
Countries:       31
Routes:          14,189
Affected routes: 674
POST obs:        270,780
PRE obs:         512,277


## 10. Save

In [15]:
route_panel_nc.to_parquet(CLEAN_DIR / 'route_panel_nc_co2.parquet', index=False)
df_co2.to_parquet(CLEAN_DIR / 'route_panel_full_co2.parquet', index=False)

print(f'route_panel_nc_co2:   {route_panel_nc.shape}')
print(f'route_panel_full_co2: {df_co2.shape}')

route_panel_nc_co2:   (783057, 46)
route_panel_full_co2: (885377, 44)
